In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "c830640b",
   "metadata": {},
   "source": [
    "# 01 · Bronze (local + pandas) — escrevendo no MinIO\n",
    "\n",
    "```\n",
    "data/lote1 ──► landing (bucket MinIO) ──► BRONZE (Parquet no bucket warehouse)\n",
    "```\n",
    "\n",
    "Aqui **não há Spark**: o pandas lê os arquivos da landing no MinIO, acrescenta as 3 colunas de rastreio (`_ingest_ts`, `_source_file`, `_batch_id`) e grava **Parquet** no bucket `warehouse`.\n",
    "\n",
    "**Diferença para a versão Spark:** estas são pastas de arquivos Parquet por lote, **não tabelas Iceberg**. Iceberg precisa de um motor como o Spark para escrever os metadados. A ideia da Bronze é a mesma: guardar como veio, com rastreabilidade, e poder reexecutar sem duplicar.\n",
    "\n",
    "**Antes de começar:** o MinIO precisa estar no ar (`docker compose up -d`) e o ambiente precisa ter `pandas`, `pyarrow` e `boto3`."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "f5adf019",
   "metadata": {},
   "source": [
    "## Passo 0 · Conexão com o MinIO e criação dos buckets"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "7163b07a",
   "metadata": {},
   "outputs": [],
   "source": [
    "import io\n",
    "import os\n",
    "from datetime import datetime, timezone\n",
    "from pathlib import Path\n",
    "\n",
    "import boto3\n",
    "import pandas as pd\n",
    "\n",
    "ENDPOINT = os.environ.get(\"MINIO_ENDPOINT\", \"http://localhost:9000\")\n",
    "USUARIO = os.environ.get(\"MINIO_USER\", \"admin\")\n",
    "SENHA = os.environ.get(\"MINIO_PASSWORD\", \"admin12345\")\n",
    "BATCH = \"lote1\"\n",
    "PREFIXO_LANDING = f\"mercadonordeste/{BATCH}\"\n",
    "\n",
    "# a pasta data pode estar ao lado do notebook ou uma pasta acima\n",
    "DATA = next(p for p in (Path(\"data\"), Path(\"../data\")) if (p / BATCH).exists()) / BATCH\n",
    "\n",
    "s3 = boto3.client(\"s3\", endpoint_url=ENDPOINT, aws_access_key_id=USUARIO,\n",
    "                  aws_secret_access_key=SENHA, region_name=\"us-east-1\")\n",
    "\n",
    "existentes = {b[\"Name\"] for b in s3.list_buckets()[\"Buckets\"]}\n",
    "for bucket in (\"landing\", \"warehouse\"):\n",
    "    if bucket not in existentes:\n",
    "        s3.create_bucket(Bucket=bucket)\n",
    "print(\"buckets:\", sorted(b[\"Name\"] for b in s3.list_buckets()[\"Buckets\"]))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a941a67c",
   "metadata": {},
   "source": [
    "## Passo 1 · Os arquivos chegam na landing\n",
    "Enviamos os 4 arquivos como um sistema de origem faria, pela API S3. A landing guarda o dado **exatamente como veio**."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "fbd7d603",
   "metadata": {},
   "outputs": [],
   "source": [
    "for arquivo in sorted(DATA.iterdir()):\n",
    "    s3.upload_file(str(arquivo), \"landing\", f\"{PREFIXO_LANDING}/{arquivo.name}\")\n",
    "\n",
    "for o in s3.list_objects_v2(Bucket=\"landing\", Prefix=PREFIXO_LANDING)[\"Contents\"]:\n",
    "    print(f\"{o['Size']:>10,} bytes  s3://landing/{o['Key']}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "1793f15a",
   "metadata": {},
   "source": [
    "Abra o console do MinIO em <http://localhost:9001> (usuário e senha do seu `.env`) e veja o bucket **landing**.\n",
    "É um *object storage*: não há pastas de verdade, só chaves com `/` no nome."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "7a3b56b9",
   "metadata": {},
   "source": [
    "### Experimento: o que é uma \"pasta\" aqui?\n",
    "\n",
    "Vamos pedir ao MinIO as chaves de três formas e fazer o experimento de **renomear**.\n",
    "Antes de rodar, pense: se não existem pastas, como o console mostra `mercadonordeste` e `lote1` como pastas?"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "85e87884",
   "metadata": {},
   "outputs": [],
   "source": [
    "# função auxiliar: lista (chave, tamanho) de todos os objetos de um prefixo (com paginação)\n",
    "def listar(bucket, prefixo=\"\"):\n",
    "    achados = []\n",
    "    for pagina in s3.get_paginator(\"list_objects_v2\").paginate(Bucket=bucket, Prefix=prefixo):\n",
    "        achados += [(o[\"Key\"], o[\"Size\"]) for o in pagina.get(\"Contents\", [])]\n",
    "    return achados\n",
    "\n",
    "\n",
    "# 1) \"Pastas\" são só prefixos: pedimos ao MinIO para agrupar as chaves pelo \"/\"\n",
    "resp = s3.list_objects_v2(Bucket=\"landing\", Prefix=\"mercadonordeste/\", Delimiter=\"/\")\n",
    "print(\"Prefixos:\", [p[\"Prefix\"] for p in resp.get(\"CommonPrefixes\", [])])\n",
    "\n",
    "# 2) Listar por prefixo\n",
    "for chave, tam in listar(\"landing\", \"mercadonordeste/lote1/\"):\n",
    "    print(f\"{tam:>10,} bytes  {chave}\")\n",
    "\n",
    "# 3) Renomear não existe: é COPIAR e APAGAR (numa chave de teste, sem mexer nos dados)\n",
    "s3.put_object(Bucket=\"landing\", Key=\"teste/a.txt\", Body=b\"ola\")\n",
    "s3.copy_object(Bucket=\"landing\", Key=\"teste/b.txt\",\n",
    "               CopySource={\"Bucket\": \"landing\", \"Key\": \"teste/a.txt\"})\n",
    "s3.delete_object(Bucket=\"landing\", Key=\"teste/a.txt\")\n",
    "print(\"Depois do 'renomear':\", [c for c, _ in listar(\"landing\", \"teste/\")])\n",
    "s3.delete_object(Bucket=\"landing\", Key=\"teste/b.txt\")      # limpa o teste"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "d761d397",
   "metadata": {},
   "source": [
    "A \"pasta\" é só um **prefixo**: o MinIO a simula quando pedimos o agrupamento por `/`. Renomear são duas operações (copiar e apagar). Num bucket com milhões de objetos, \"renomear uma pasta\" significa copiar tudo."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "e7c3ac47",
   "metadata": {},
   "source": [
    "## Passo 2 · Landing → Bronze\n",
    "- Os CSVs são lidos como **texto** (`dtype=str`), sem tipar. Tipar é trabalho da Prata.\n",
    "- O JSON é lido sem inferência de tipos (`dtype=False`) e `itens` continua sendo uma lista.\n",
    "- **Idempotência:** antes de gravar, apagamos o que este lote já gravou. Reexecutar não duplica."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "0d37fe98",
   "metadata": {},
   "outputs": [],
   "source": [
    "INGEST_TS = datetime.now(timezone.utc)\n",
    "\n",
    "\n",
    "def ler_landing(nome):\n",
    "    return s3.get_object(Bucket=\"landing\", Key=f\"{PREFIXO_LANDING}/{nome}\")[\"Body\"].read()\n",
    "\n",
    "\n",
    "def com_rastreio(df, nome_arquivo):\n",
    "    df = df.copy()\n",
    "    df[\"_ingest_ts\"] = INGEST_TS\n",
    "    df[\"_source_file\"] = f\"s3://landing/{PREFIXO_LANDING}/{nome_arquivo}\"\n",
    "    df[\"_batch_id\"] = BATCH\n",
    "    return df\n",
    "\n",
    "\n",
    "def apagar_prefixo(bucket, prefixo):\n",
    "    for pagina in s3.get_paginator(\"list_objects_v2\").paginate(Bucket=bucket, Prefix=prefixo):\n",
    "        chaves = [{\"Key\": o[\"Key\"]} for o in pagina.get(\"Contents\", [])]\n",
    "        if chaves:\n",
    "            s3.delete_objects(Bucket=bucket, Delete={\"Objects\": chaves})\n",
    "\n",
    "\n",
    "def gravar_bronze(df, tabela):\n",
    "    prefixo = f\"bronze/{tabela}/{BATCH}/\"\n",
    "    apagar_prefixo(\"warehouse\", prefixo)          # idempotência: refaz só este lote\n",
    "    buffer = io.BytesIO()\n",
    "    df.to_parquet(buffer, index=False, engine=\"pyarrow\")\n",
    "    s3.put_object(Bucket=\"warehouse\", Key=f\"{prefixo}part-0000.parquet\", Body=buffer.getvalue())\n",
    "    return len(df)\n",
    "\n",
    "\n",
    "def ingerir_lote():\n",
    "    csv = lambda nome: pd.read_csv(io.BytesIO(ler_landing(nome)), dtype=str)\n",
    "    pedidos = pd.read_json(io.BytesIO(ler_landing(\"pedidos.jsonl\")), lines=True,\n",
    "                           dtype=False, convert_dates=False)\n",
    "    fontes = {\n",
    "        \"clientes\":   (csv(\"clientes.csv\"), \"clientes.csv\"),\n",
    "        \"produtos\":   (csv(\"produtos.csv\"), \"produtos.csv\"),\n",
    "        \"pedidos\":    (pedidos, \"pedidos.jsonl\"),\n",
    "        \"avaliacoes\": (csv(\"avaliacoes.csv\"), \"avaliacoes.csv\"),\n",
    "    }\n",
    "    return {tabela: gravar_bronze(com_rastreio(df, arq), tabela) for tabela, (df, arq) in fontes.items()}\n",
    "\n",
    "\n",
    "ingerir_lote()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "b7b059a3",
   "metadata": {},
   "source": [
    "Resultado esperado, aproximadamente: **clientes 5.000 · produtos 200 · pedidos 30.150 · avaliações 6.489**. Os pedidos incluem duplicados de propósito, e a Bronze **não** os remove."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "e3f5ddcd",
   "metadata": {},
   "source": [
    "## Passo 3 · Ler de volta e inspecionar"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "546eefbc",
   "metadata": {},
   "outputs": [],
   "source": [
    "def ler_bronze(tabela):\n",
    "    partes = []\n",
    "    for pagina in s3.get_paginator(\"list_objects_v2\").paginate(Bucket=\"warehouse\", Prefix=f\"bronze/{tabela}/\"):\n",
    "        for o in pagina.get(\"Contents\", []):\n",
    "            corpo = s3.get_object(Bucket=\"warehouse\", Key=o[\"Key\"])[\"Body\"].read()\n",
    "            partes.append(pd.read_parquet(io.BytesIO(corpo)))\n",
    "    return pd.concat(partes, ignore_index=True)\n",
    "\n",
    "\n",
    "pedidos = ler_bronze(\"pedidos\")\n",
    "print(pedidos.shape)\n",
    "print(pedidos.dtypes)\n",
    "pedidos[[\"pedido_id\", \"cliente_id\", \"status\", \"itens\", \"_batch_id\", \"_source_file\"]].head(3)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "7b40f2fe",
   "metadata": {},
   "source": [
    "Repare: `itens` continua sendo uma lista, e as colunas dos CSVs são texto. Isso é a Bronze.\n",
    "\n",
    "## Passo 4 · Onde está a tabela `pedidos`?\n",
    "\n",
    "Aqui não existe catálogo: a \"tabela\" é só uma convenção de **prefixo**, `bronze/<tabela>/<lote>/`, com um Parquet dentro. Vamos olhar o que o storage guarda de verdade."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "2a8c046d",
   "metadata": {},
   "outputs": [],
   "source": [
    "def resumo_objetos(prefixo):\n",
    "    objs = listar(\"warehouse\", prefixo)\n",
    "    return {\"objetos\": len(objs), \"KB\": round(sum(t for _, t in objs) / 1024)}\n",
    "\n",
    "\n",
    "for chave, tam in listar(\"warehouse\", \"bronze/\"):\n",
    "    print(f\"{tam:>10,}  s3://warehouse/{chave}\")\n",
    "print(resumo_objetos(\"bronze/pedidos/\"))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "6e51ab88",
   "metadata": {},
   "source": [
    "Confira também no console do MinIO (<http://localhost:9001>), no bucket **warehouse**, pasta `bronze`.\n",
    "\n",
    "**Para discutir:** não há `metadata/`. Nada diz quais arquivos fazem parte da tabela além do nome do prefixo. Se alguém colocar um arquivo qualquer em `bronze/pedidos/lote1/`, a função `ler_bronze` vai lê-lo como se fosse dado. Na versão Spark/Iceberg, a pasta `metadata/` registra exatamente quais arquivos formam cada versão da tabela."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "268d17e3",
   "metadata": {},
   "source": [
    "## Passo 5 · Idempotência\n",
    "Rode a ingestão de novo. A contagem de linhas **não** dobra. Vamos conferir também o que aconteceu com os **objetos**."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "46a773b0",
   "metadata": {},
   "outputs": [],
   "source": [
    "def objetos(prefixo):\n",
    "    achados = {}\n",
    "    for pagina in s3.get_paginator(\"list_objects_v2\").paginate(Bucket=\"warehouse\", Prefix=prefixo):\n",
    "        for o in pagina.get(\"Contents\", []):\n",
    "            achados[o[\"Key\"]] = o[\"LastModified\"]\n",
    "    return achados\n",
    "\n",
    "\n",
    "prefixo = \"bronze/pedidos/\"\n",
    "linhas_antes, obj_antes = len(ler_bronze(\"pedidos\")), objetos(prefixo)\n",
    "\n",
    "ingerir_lote()                                    # reexecuta o MESMO lote\n",
    "\n",
    "linhas_depois, obj_depois = len(ler_bronze(\"pedidos\")), objetos(prefixo)\n",
    "\n",
    "print(\"Linhas :\", linhas_antes, \"->\", linhas_depois, \"| idempotente:\", linhas_antes == linhas_depois)\n",
    "print(\"Objetos:\", len(obj_antes), \"->\", len(obj_depois))\n",
    "for chave in obj_depois:\n",
    "    print(f\"  {chave}\\n    gravado em {obj_antes.get(chave)}  ->  {obj_depois[chave]}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "b2d57ce3",
   "metadata": {},
   "source": [
    "### O que aconteceu?\n",
    "\n",
    "As **linhas** não dobraram e o **número de objetos** também não mudou, mas a data de gravação do objeto mudou (se vir a mesma data, rode a célula de novo). O storage não edita arquivos: apagamos o objeto antigo e gravamos um novo no mesmo lugar.\n",
    "\n",
    "Isso é idempotência por **sobrescrita**: simples, mas o passado deixa de existir. Na versão com Spark e Iceberg os objetos **aumentam** a cada reexecução, porque o Iceberg registra uma nova versão da tabela e mantém os arquivos antigos, o que permite voltar no tempo.\n",
    "\n",
    "**Pergunta:** se a ingestão de hoje tivesse gravado um lote com erro, como você recuperaria o lote de ontem aqui? E com Iceberg?"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "8f00dbdd",
   "metadata": {},
   "source": [
    "## Qualidade e governança neste notebook\n",
    "- **Qualidade:** a Bronze é a prova do que a origem enviou, por isso não se limpa. Se a regra de limpeza estiver errada, reprocessa-se a partir daqui.\n",
    "- **Linhagem:** pegue um `pedido_id` e responda de qual arquivo e de qual lote ele veio (`_source_file`, `_batch_id`).\n",
    "- **Acesso:** a landing e o warehouse usam a mesma credencial `admin`. Quem deveria poder gravar na landing? Quem deveria só ler a Bronze?\n",
    "- **Armazenamento x processamento:** pare o MinIO (`docker compose stop`) e rode o notebook: o pandas continua instalado, mas não há onde ler nem gravar. O dado vive no storage, o código só passa por ele.\n",
    "- **Segredos:** a senha vem de variável de ambiente (`MINIO_PASSWORD`), com o padrão de laboratório. Em produção ela ficaria num cofre de segredos.\n",
    "\n",
    "## Checkpoint\n",
    "- [ ] Os 4 arquivos estão em `s3://landing/mercadonordeste/lote1/`.\n",
    "- [ ] Há um Parquet por tabela em `s3://warehouse/bronze/<tabela>/lote1/`.\n",
    "- [ ] Reexecutar não duplicou, e você explica por que o número de objetos não mudou."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}